In [ ]:
import h5py
import random
import os

import cmocean
import numpy as np
from matplotlib import pyplot as plt

# Visualize ESDC LST samples directly from the HDF5 source (pre-LMDB).
# Each sample is a spatiotemporal patch with 5 time steps.
# NaN pixels indicate gaps (missing observations).
# The middle time step (t=2) is the reconstruction target.

_esdc_data_dir = os.environ.get(
    "ESDC_DATA_DIR",
    "../../../esdc/data/lst_no_syn_gaps",
)

h5_path = os.path.join(_esdc_data_dir, "dataset.h5")

n_samples = 10
n_timesteps = 5
seed = None  # set to an int for reproducibility
TARGET_T = n_timesteps // 2
VAL_RANGE = (180, 360)
CMAP = cmocean.cm.matter_r
VMIN, VMAX = 273.15, 323.15

if seed is not None:
    random.seed(seed)

with h5py.File(h5_path, "r") as f:
    n_total = f["X_train"].shape[0]
    indices = sorted(random.sample(range(n_total), min(n_samples, n_total)))

    fig, axes = plt.subplots(
        len(indices), n_timesteps, figsize=(20, 4 * len(indices))
    )
    if len(indices) == 1:
        axes = [axes]

    for i, idx in enumerate(indices):
        x = f["X_train"][idx]  # (T, H, W, C=2)
        lst = x[:, :, :, 0].astype(np.float32)  # (T, H, W), LST channel
        lst = lst * (VAL_RANGE[1] - VAL_RANGE[0]) + VAL_RANGE[0]  # → Kelvin

        for t in range(n_timesteps):
            ax = axes[i][t]
            ax.matshow(lst[t], cmap=CMAP, vmin=VMIN, vmax=VMAX)

            H, W = lst[t].shape
            frac = 100.0 * float(np.isnan(lst[t]).sum()) / float(lst[t].size)
            title = f"Sample {i + 1} (idx={idx}), t={t}\n{H}\u00d7{W}  gaps={frac:.1f}%"
            if t == TARGET_T:
                title += " (target)"
            ax.set_title(title)
            ax.set_xticks([])
            ax.set_yticks([])

plt.tight_layout()
plt.show()